# Task 02 — Few-Shot Entity Extraction

Build a few-shot extraction pipeline that pulls structured info from customer messages.

**Target output**: `{"product": str, "issue": str | null, "sentiment": "positive" | "negative"}`

**You will implement**:
1. `FEW_SHOT_EXAMPLES` — at least 3 labeled examples (covers positive + negative + null issue)
2. `EXTRACTION_SYSTEM_PROMPT` — system instructions for the extractor
3. `build_extraction_prompt(text, examples)` → `str` — few-shot user message
4. `extract_entities(client, text)` → `dict` — full pipeline with real API call
5. Accuracy/format test on all 10 extraction samples

## Setup

In [ ]:
from openai import OpenAI
import json, os

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "extraction_samples.json")) as f:
    extraction_samples = json.load(f)

def parse_json_safe(text: str) -> dict | None:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print(f"✓ Setup complete. {len(extraction_samples)} extraction samples loaded.")

## Task 2.1 — Define Few-Shot Examples

Create `FEW_SHOT_EXAMPLES`: list of dicts with `"input"` and `"output"` keys.

Requirements:
- At least 3 examples
- Cover both `"positive"` and `"negative"` sentiment
- At least one example with `"issue": null` (satisfied customer)
- All `"output"` values must be valid JSON with `product`, `issue`, `sentiment`

In [ ]:
# YOUR CODE HERE
FEW_SHOT_EXAMPLES = []

# TEST — Do not modify
assert len(FEW_SHOT_EXAMPLES) >= 3, f"Need >= 3 examples, got {len(FEW_SHOT_EXAMPLES)}"

for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    assert "input"  in ex, f"Example {i} missing 'input'"
    assert "output" in ex, f"Example {i} missing 'output'"
    parsed = json.loads(ex["output"])  # raises if invalid JSON
    assert "product"   in parsed, f"Example {i} output missing 'product'"
    assert "sentiment" in parsed, f"Example {i} output missing 'sentiment'"
    assert parsed["sentiment"] in ("positive", "negative"),         f"Example {i} sentiment must be 'positive' or 'negative'"

sentiments = [json.loads(ex["output"])["sentiment"] for ex in FEW_SHOT_EXAMPLES]
assert "positive" in sentiments, "Include at least one positive example"
assert "negative" in sentiments, "Include at least one negative example"

null_issue_count = sum(1 for ex in FEW_SHOT_EXAMPLES if json.loads(ex["output"]).get("issue") is None)
assert null_issue_count >= 1, "Include at least one example where issue is null"

print(f"✓ Task 2.1 passed ({len(FEW_SHOT_EXAMPLES)} examples)")

## Task 2.2 — Design the Extraction System Prompt

Write `EXTRACTION_SYSTEM_PROMPT` that instructs the model to extract
`product`, `issue` (null if none), and `sentiment` from customer messages.

In [ ]:
# YOUR CODE HERE
EXTRACTION_SYSTEM_PROMPT = "..."

# TEST — Do not modify
assert isinstance(EXTRACTION_SYSTEM_PROMPT, str)
assert len(EXTRACTION_SYSTEM_PROMPT.strip()) >= 50
for kw in ["product", "sentiment", "json"]:
    assert kw in EXTRACTION_SYSTEM_PROMPT.lower(), f"Prompt must mention '{kw}'"
assert "positive" in EXTRACTION_SYSTEM_PROMPT.lower() and "negative" in EXTRACTION_SYSTEM_PROMPT.lower(),     "Prompt must specify both 'positive' and 'negative' as valid sentiment values"

print("✓ Task 2.2 passed")

## Task 2.3 — Implement build_extraction_prompt()

```python
def build_extraction_prompt(text: str, examples: list[dict]) -> str:
```

Returns a string that contains all examples AND the query `text`.

In [ ]:
# YOUR CODE HERE
def build_extraction_prompt(text: str, examples: list[dict]) -> str:
    ...

# TEST — Do not modify
test_text = "My Sony WF-1000XM5 earbuds have terrible battery life."
prompt = build_extraction_prompt(test_text, FEW_SHOT_EXAMPLES)

assert isinstance(prompt, str)
assert test_text in prompt, "Prompt must contain the query text"
assert len(prompt) > len(test_text), "Prompt must include examples beyond the query text"
for ex in FEW_SHOT_EXAMPLES[:2]:
    assert ex["input"] in prompt, f"Prompt must contain example: {ex['input'][:40]!r}"

print("✓ Task 2.3 passed")

## Task 2.4 — Implement extract_entities()

```python
def extract_entities(client, text: str) -> dict:
```

Full pipeline: `build_extraction_prompt` → API call → parse JSON → return dict.

In [ ]:
# YOUR CODE HERE
def extract_entities(client, text: str) -> dict:
    ...

# TEST — real API call on a negative sentiment sample
sample = extraction_samples[0]  # MacBook Pro keyboard issue
result = extract_entities(client, sample["text"])

assert isinstance(result, dict), f"extract_entities must return dict, got {type(result)}"
assert "product"   in result, f"Result must have 'product'. Got keys: {list(result.keys())}"
assert "sentiment" in result, f"Result must have 'sentiment'. Got keys: {list(result.keys())}"
assert result["sentiment"] in ("positive", "negative"),     f"sentiment must be 'positive' or 'negative', got {result['sentiment']!r}"
assert isinstance(result.get("product"), str) and len(result["product"]) > 0,     "product must be a non-empty string"

print(f"✓ Task 2.4 passed")
print(f"  Text:      {sample['text'][:70]}...")
print(f"  Got:       {result}")
print(f"  Expected:  {sample['expected']}")

In [ ]:
# TEST — positive sentiment + null issue (sample 2: Samsung Galaxy S24 happy customer)
positive_sample = extraction_samples[2]
result_pos = extract_entities(client, positive_sample["text"])

assert result_pos.get("sentiment") == "positive",     f"Expected 'positive' for satisfied customer, got {result_pos.get('sentiment')!r}"
assert result_pos.get("issue") is None or result_pos.get("issue") == "",     f"Expected null/empty issue for positive feedback, got {result_pos.get('issue')!r}"

print(f"✓ Task 2.4b passed — positive sentiment + null issue handled")
print(f"  Text: {positive_sample['text'][:70]}")
print(f"  Got:  {result_pos}")

## Task 2.5 — Format Accuracy on All 10 Samples

Run `extract_entities` on all 10 samples. Every result must:
- Have `product`, `issue`, `sentiment` keys
- Have `sentiment` in `{"positive", "negative"}`
- Have non-empty `product` string

In [ ]:
# TEST — real API calls on all 10 extraction samples
format_ok = 0
sentiment_correct = 0

for s in extraction_samples:
    result = extract_entities(client, s["text"])
    has_keys = all(k in result for k in ["product", "sentiment"])
    valid_sentiment = result.get("sentiment") in ("positive", "negative")
    nonempty_product = isinstance(result.get("product"), str) and len(result["product"]) > 0

    if has_keys and valid_sentiment and nonempty_product:
        format_ok += 1
    if result.get("sentiment") == s["expected"]["sentiment"]:
        sentiment_correct += 1

    print(f"  product={result.get('product')!r:30} sentiment={result.get('sentiment'):10} "
          f"expected_sentiment={s['expected']['sentiment']}")

format_rate = format_ok / len(extraction_samples)
sentiment_acc = sentiment_correct / len(extraction_samples)
print(f"\nFormat correct:     {format_ok}/{len(extraction_samples)} = {format_rate:.0%}")
print(f"Sentiment accuracy: {sentiment_correct}/{len(extraction_samples)} = {sentiment_acc:.0%}")

assert format_rate == 1.0, f"All results must have correct format. Got {format_ok}/{len(extraction_samples)}"
assert sentiment_acc >= 0.80, f"Sentiment accuracy {sentiment_acc:.0%} < 80% — improve your prompt or examples"
print("\n✓ Task 2.5 passed")

## Done

In [ ]:
print("\n✓ All task_02 tests passed!")